# TALOS — Reindex de Embeddings (Colab Offload)

**Corpus:** ~90 067 chunks | **Modelo:** multilingual-e5-small ONNX | **Output:** vector_store.db

## Antes de ejecutar, subí a Drive (carpeta talos_reindex):
- knowledge_store.db (290 MB) desde data/web_state/
- model.onnx (449 MB) desde models/multilingual-e5-small/onnx/
- tokenizer.json (17 MB) desde models/multilingual-e5-small/onnx/

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive montado")

In [ ]:
import os

DRIVE_FOLDER = "/content/drive/MyDrive/talos_reindex"  # EDITAR si es necesario
KNOWLEDGE_DB = f"{DRIVE_FOLDER}/knowledge_store.db"
MODEL_ONNX   = f"{DRIVE_FOLDER}/model.onnx"
TOKENIZER    = f"{DRIVE_FOLDER}/tokenizer.json"
OUTPUT_DB    = f"{DRIVE_FOLDER}/vector_store.db"
SHARD_DIR    = "/content/vec_shards"

for path, label in [(KNOWLEDGE_DB, "knowledge_store.db"), (MODEL_ONNX, "model.onnx"), (TOKENIZER, "tokenizer.json")]:
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else -1
    print(f"{label}: {size:.0f} MB" if size > 0 else f"{label}: NO ENCONTRADO")

os.makedirs(SHARD_DIR, exist_ok=True)

In [ ]:
!pip install -q --upgrade --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ onnxruntime-gpu tokenizers numpy tqdm

In [ ]:
import onnxruntime as ort
available = ort.get_available_providers()
print("Providers:", available)
PROVIDER = "CUDAExecutionProvider" if "CUDAExecutionProvider" in available else "CPUExecutionProvider"
print(f"Usando: {PROVIDER}")

In [ ]:
import sqlite3
con = sqlite3.connect(f"file:{KNOWLEDGE_DB}?mode=ro", uri=True)
rows = con.execute("SELECT id, source_id, content FROM chunks ORDER BY id").fetchall()
chunks = [(r[0], r[1], r[2] or "") for r in rows]
src_to_nbs = {}
for src_id, nb_id in con.execute("SELECT source_id, notebook_id FROM notebook_sources").fetchall():
    src_to_nbs.setdefault(src_id, []).append(nb_id)
con.close()
print(f"Chunks: {len(chunks):,} | Fuentes: {len(src_to_nbs):,}")

In [ ]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(TOKENIZER)
tokenizer.enable_truncation(max_length=512)
tokenizer.enable_padding(pad_id=0, pad_token="<pad>", length=None)

opts = ort.SessionOptions()
opts.intra_op_num_threads = 4
opts.inter_op_num_threads = 1
session = ort.InferenceSession(MODEL_ONNX, sess_options=opts, providers=[PROVIDER])
print(f"Modelo cargado con {PROVIDER}")

DIMENSIONS = 384

def embed_passages(texts):
    formatted = [f"passage: {t.strip()}" for t in texts]
    enc = tokenizer.encode_batch(formatted)
    ids  = np.array([e.ids            for e in enc], dtype=np.int64)
    mask = np.array([e.attention_mask for e in enc], dtype=np.int64)
    tids = np.array([e.type_ids       for e in enc], dtype=np.int64)
    out = session.run(None, {"input_ids": ids, "attention_mask": mask, "token_type_ids": tids})
    lhs = out[0]
    m   = np.expand_dims(mask, -1).astype(np.float32)
    emb = np.sum(lhs * m, axis=1) / np.clip(m.sum(axis=1), 1e-9, None)
    nrm = np.linalg.norm(emb, axis=1, keepdims=True)
    return (emb / np.clip(nrm, 1e-9, None)).tolist()

assert len(embed_passages(["test"])[0]) == DIMENSIONS
print("Smoke test OK")

In [ ]:
import struct, time
from tqdm.notebook import tqdm

BATCH_SIZE  = 128
PROVIDER_ID = "local_onnx"
MODEL_ID    = "multilingual-e5-small"

shard_path = f"{SHARD_DIR}/shard_0.db"
conn = sqlite3.connect(shard_path)
conn.execute("PRAGMA journal_mode = OFF;")
conn.execute("PRAGMA synchronous = OFF;")
conn.execute("""CREATE TABLE IF NOT EXISTS chunk_embeddings (
    chunk_id TEXT NOT NULL, notebook_id TEXT NOT NULL, source_id TEXT NOT NULL,
    provider_id TEXT NOT NULL, model_id TEXT NOT NULL, dimensions INTEGER NOT NULL,
    vector BLOB NOT NULL, PRIMARY KEY (chunk_id, notebook_id));""")

already_done = {r[0] for r in conn.execute("SELECT DISTINCT chunk_id FROM chunk_embeddings")}
pending = [c for c in chunks if c[0] not in already_done]
print(f"Resume: {len(already_done):,} ya listos, {len(pending):,} pendientes")

total_rows, t0, done = 0, time.time(), len(already_done)
with tqdm(total=len(chunks), initial=done, unit="chunks", desc="Embedding") as pbar:
    for i in range(0, len(pending), BATCH_SIZE):
        batch = pending[i:i+BATCH_SIZE]
        vectors = embed_passages([c[2] for c in batch])
        payload = []
        for (cid, sid, _), vec in zip(batch, vectors):
            for nb in src_to_nbs.get(sid, []):
                payload.append((cid, nb, sid, PROVIDER_ID, MODEL_ID, DIMENSIONS, struct.pack(f"{DIMENSIONS}f", *vec)))
        conn.executemany("INSERT OR REPLACE INTO chunk_embeddings VALUES (?,?,?,?,?,?,?)", payload)
        conn.commit()
        total_rows += len(payload)
        done += len(batch)
        elapsed = time.time() - t0
        rate = (done - len(already_done)) / max(elapsed, 1e-6)
        remaining = (len(pending) - (done - len(already_done))) / max(rate, 1e-6)
        pbar.update(len(batch))
        pbar.set_postfix(rows=f"{total_rows:,}", cps=f"{rate:.0f}", eta=f"{remaining/60:.1f}m")
conn.close()
print(f"Embedding completo: {total_rows:,} filas en {(time.time()-t0)/60:.1f} min")

In [ ]:
final_conn = sqlite3.connect(OUTPUT_DB)
final_conn.execute("PRAGMA journal_mode = WAL;")
final_conn.execute("PRAGMA synchronous = NORMAL;")
final_conn.execute("DROP TABLE IF EXISTS chunk_embeddings;")
final_conn.execute("""CREATE TABLE chunk_embeddings (
    chunk_id TEXT NOT NULL, notebook_id TEXT NOT NULL, source_id TEXT NOT NULL,
    provider_id TEXT NOT NULL, model_id TEXT NOT NULL, dimensions INTEGER NOT NULL,
    vector BLOB NOT NULL, PRIMARY KEY (chunk_id, notebook_id));""")
final_conn.execute("CREATE INDEX IF NOT EXISTS idx_vec_notebook ON chunk_embeddings(notebook_id);")
final_conn.execute("CREATE INDEX IF NOT EXISTS idx_vec_source ON chunk_embeddings(source_id);")
final_conn.execute(f"ATTACH DATABASE '{shard_path}' AS shard0")
final_conn.execute("BEGIN")
final_conn.execute("INSERT OR REPLACE INTO chunk_embeddings SELECT chunk_id, notebook_id, source_id, provider_id, model_id, dimensions, vector FROM shard0.chunk_embeddings")
final_conn.commit()
final_conn.execute("DETACH DATABASE shard0")
counts = final_conn.execute("SELECT notebook_id, COUNT(*) FROM chunk_embeddings GROUP BY notebook_id ORDER BY 2 DESC").fetchall()
total_final = final_conn.execute("SELECT COUNT(*) FROM chunk_embeddings").fetchone()[0]
final_conn.close()
size_mb = os.path.getsize(OUTPUT_DB) / 1e6
print(f"vector_store.db: {size_mb:.1f} MB | {total_final:,} filas")
for nb, n in counts: print(f"  {nb}: {n:,}")

## Para reemplazar localmente:



O si usas Drive Sync, simplemente copiá el archivo desde la carpeta sincronizada.